# Image Classification with TensorFlow

This notebook trains and compares two CNN architectures on CIFAR-10.
It includes dataset loading, augmentation, model definitions, training, evaluation, and visualization.

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

dataset_name = 'cifar10'  # change to 'mnist' to use MNIST instead

def load_data(name):
    if name == 'mnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
        x_train = np.expand_dims(x_train, axis=-1)
        x_test = np.expand_dims(x_test, axis=-1)
        num_classes = 10
    elif name == 'cifar10':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
        y_train = y_train.flatten()
        y_test = y_test.flatten()
        num_classes = 10
    else:
        raise ValueError('Unsupported dataset: ' + name)

    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    return (x_train, y_train), (x_test, y_test), num_classes

(x_train, y_train), (x_test, y_test), num_classes = load_data(dataset_name)

print('Train shape:', x_train.shape, 'Test shape:', x_test.shape)
print('Classes:', num_classes)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def build_baseline_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

def build_deeper_model(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name='deeper_cnn')

input_shape = x_train.shape[1:]
baseline_model = build_baseline_model(input_shape, num_classes)
deeper_model = build_deeper_model(input_shape, num_classes)

baseline_model.summary()
deeper_model.summary()

def compile_model(model):
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

compile_model(baseline_model)
compile_model(deeper_model)

def train_model(model, x_train, y_train, x_val, y_val, epochs=15, batch_size=64):
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=2
    )
    return history

val_split = 0.1
val_count = int(len(x_train) * val_split)
x_val = x_train[:val_count]
y_val = y_train[:val_count]
x_train_sub = x_train[val_count:]
y_train_sub = y_train[val_count:]

history_baseline = train_model(baseline_model, x_train_sub, y_train_sub, x_val, y_val)
history_deeper = train_model(deeper_model, x_train_sub, y_train_sub, x_val, y_val)

def plot_history(history, label):
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['accuracy'], label=f'{label} train')
    plt.plot(history.history['val_accuracy'], label=f'{label} val')
    plt.title(f'{label} accuracy')
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(history_baseline, 'Baseline CNN')
plot_history(history_deeper, 'Deeper CNN')

test_loss_b, test_acc_b = baseline_model.evaluate(x_test, y_test, verbose=2)
test_loss_d, test_acc_d = deeper_model.evaluate(x_test, y_test, verbose=2)

print(f'Baseline test accuracy: {test_acc_b:.4f}')
print(f'Deeper test accuracy: {test_acc_d:.4f}')

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

def show_predictions(model, x_data, y_data, count=12):
    predictions = model.predict(x_data[:count])
    predicted_labels = np.argmax(predictions, axis=1)

    plt.figure(figsize=(12, 8))
    for i in range(count):
        plt.subplot(3, 4, i + 1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        plt.imshow(x_data[i])
        color = 'blue' if predicted_labels[i] == y_data[i] else 'red'
        plt.xlabel(f'pred: {class_names[predicted_labels[i]]}
actual: {class_names[y_data[i]]}', color=color)
    plt.tight_layout()
    plt.show()

show_predictions(deeper_model, x_test, y_test)

y_pred = np.argmax(deeper_model.predict(x_test), axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix for Deeper CNN')
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))

deeper_model.save('deeper_cnn_cifar10.h5')
print('Saved deeper_cnn_cifar10.h5')